# Hybrid CNN-ViT for Edge Devices: MobileNetV2 vs MobileViT-XS on PlantVillage

**Experiments and results notebook**: data preparation → training → evaluation → deployability → qualitative analysis.

**How to run on Google Colab (free tier)**
1. `Runtime → Change runtime type → T4 GPU`.
2. Once only: put the Kaggle PlantVillage zip (`mohitsingh1804/plantvillage`) in Google Drive at
   `MyDrive/cnn-vit-research/data/` (any file name ending in `.zip`).
3. `Runtime → Run all`. Keep the browser tab open.

**Everything is saved to `MyDrive/cnn-vit-research/`** (checkpoints after every epoch, best weights, histories,
predictions, tables and figures). If Colab disconnects, just reconnect and **Run all** again:
finished runs are skipped and an unfinished run continues from its last epoch.

Each run produces the same results when resumed, because data order, augmentation and dropout
are seeded by (seed, epoch). Small differences can still come from GPU numerics.

## 0. Setup

In [ ]:
# --- settings you might change ------------------------------------------------------------
BRANCH = "feature/training"            # git branch with the code
DRIVE_DIR = "/content/drive/MyDrive/cnn-vit-research"
RUN_ABLATION = True                    # E6: 2 extra training runs (after the 6 main runs)
RUN_ROBUSTNESS = True                  # E5: inference only, about 10 minutes
SMOKE = False                          # True = tiny synthetic data for a quick test, not for results

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/cnn-vit-research-"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/SilverUmer21/cnn-vit-research-.git", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "scikit-learn", "scipy"], check=True)
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, REPO)
print("repo:", REPO, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import torch, transformers, platform
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | python {platform.python_version()}")
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (training on CPU is far too slow for the real data)")

### Data: copy the dataset from Drive to the fast local disk
Reading 54k small files directly from Drive is very slow, so the zip is copied and unpacked locally once per session (about 2 minutes).

In [ ]:
import glob, shutil, zipfile
from pathlib import Path

def find_data_root(base):
    # the folder that contains both train/ and val/ (the Kaggle 'PlantVillage' folder)
    for p in sorted(Path(base).rglob("train")):
        if p.is_dir() and (p.parent / "val").is_dir():
            return p.parent
    return None

if SMOKE:
    from src.experiments.smoke import make_smoke_data
    LOCAL = Path("/content/smoke" if IN_COLAB else "/tmp/plantvillage_smoke")
    make_smoke_data("splits/manifest.csv", LOCAL / "PlantVillage", LOCAL / "manifest.csv")
    DATA_ROOT, MANIFEST, PERSIST = LOCAL / "PlantVillage", LOCAL / "manifest.csv", LOCAL / "persist"
else:
    LOCAL = Path("/content/data")
    DATA_ROOT = find_data_root(LOCAL) if LOCAL.exists() else None
    if DATA_ROOT is None:
        zips = sorted(glob.glob(f"{DRIVE_DIR}/data/*.zip"))
        assert zips, f"Put the PlantVillage zip in {DRIVE_DIR}/data/ first"
        LOCAL.mkdir(parents=True, exist_ok=True)
        print("copying", zips[0]); shutil.copy(zips[0], "/content/plantvillage.zip")
        print("unzipping ..."); zipfile.ZipFile("/content/plantvillage.zip").extractall(LOCAL)
        os.remove("/content/plantvillage.zip")
        DATA_ROOT = find_data_root(LOCAL)
    MANIFEST, PERSIST = Path(REPO) / "splits/manifest.csv", Path(DRIVE_DIR)
print("data root:", DATA_ROOT)

In [ ]:
import pandas as pd
from src.experiments.config import Paths, TrainConfig, BenchmarkConfig, SEEDS, main_runs, ablation_runs, run_name, MODELS, MODEL_ORDER

paths = Paths(repo=REPO, persist=PERSIST, data_root=DATA_ROOT, manifest=MANIFEST).make()
manifest = pd.read_csv(paths.manifest)
missing = [p for p in manifest["path"] if not (paths.data_root / p).exists()]
assert not missing, f"{len(missing)} images from the manifest are missing, e.g. {missing[:3]}"
print(f"all {len(manifest):,} manifest images found; results go to {paths.persist}")

CFG = TrainConfig()
if SMOKE:
    CFG = CFG.with_overrides(epochs=2, batch_size=16, eval_batch_size=32, lr=1e-3)
SEED_LIST = SEEDS if not SMOKE else (42,)
print(CFG)

## 1. Data preparation (summary)
The audit, duplicate grouping and split were done by `src/dataset/` (see `reports/`). The split is fixed in
`splits/manifest.csv`: 54,304 images (one non-leaf image removed), pooled from the shipped train/val folders,
re-split 80/10/10, stratified by class, with byte-identical and near-duplicate images grouped so a group never spans two splits.
Image recipes are read from each checkpoint's Hugging Face preprocessor config: MobileNetV2 224², RGB, mean/std 0.5;
MobileViT-XS 256², BGR, no mean/std. Both use the same augmentation (flip, ±15° rotation, brightness/contrast 0.2).

In [ ]:
from src.experiments import plots as P, latex_tables as T, analysis as A
print(manifest.groupby("split").agg(images=("path", "size"), groups=("group_id", "nunique")).loc[["train", "val", "test"]])
groups_per_split = manifest.groupby("group_id")["split"].nunique()
print("groups spanning more than one split:", int((groups_per_split > 1).sum()))
A.write_tex(T.dataset_split(manifest), paths.tables / "split.tex")
P.class_distribution(manifest, paths.figures)
P.sample_grid(manifest, paths.data_root, paths.figures);

In [ ]:
# loader checks written by Member 1 (shapes, value ranges, BGR order, augmentation, seeding)
if not SMOKE:
    r = subprocess.run([sys.executable, "-m", "src.dataset.check_loaders", str(paths.manifest), str(paths.data_root)],
                       capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-1500:])

## 2. Training (E1 runs, then the E6 ablation runs)
Protocol (identical for both models, from the proposal): ImageNet-pretrained weights, new 38-class head, AdamW
(lr 1e-4, weight decay 1e-4), cosine annealing over 20 epochs, batch 32, cross-entropy, early stopping after 5 epochs
without validation-loss improvement, best checkpoint by validation loss. FP16 mixed precision on the GPU for speed (same for both).
Three seeds per model (42, 43, 44) for mean ± std; runs are interleaved by seed so paired results exist early.

In [ ]:
from src.experiments.train import train_run

def log(msg):
    print(msg, flush=True)

todo = main_runs(SEED_LIST) + (ablation_runs() if RUN_ABLATION and not SMOKE else [])
for i, (m, s, v) in enumerate(todo, 1):
    log(f"\n=== run {i}/{len(todo)}: {run_name(m, s, v)} ===")
    train_run(paths, m, s, v, CFG, device=DEVICE, log=log)
log("\nALL TRAINING RUNS FINISHED")

## 3. E1: Classification performance

In [ ]:
runs, preds = A.load_runs(paths)
main = A.main_results(runs)
per_seed = A.per_seed_table(runs)
stats = A.paired_statistics(preds)
cost = A.training_cost(runs)
for name, df in [("main", main), ("per_seed", per_seed), ("stats", stats), ("training_cost", cost)]:
    df.to_csv(paths.tables / f"{name}.csv", index=False)
A.write_tex(T.main_results(main), paths.tables / "main.tex")
A.write_tex(T.per_seed(per_seed), paths.tables / "per_seed.tex")
A.write_tex(T.statistics(stats), paths.tables / "stats.tex")
A.write_tex(T.training_cost(cost), paths.tables / "training_cost.tex")
display(main.round(3)); display(per_seed.round(4)); display(stats.round(4)); display(cost.round(2))

In [ ]:
P.learning_curves(paths, runs, paths.figures);

In [ ]:
pc = A.per_class_table(preds)
pc.to_csv(paths.tables / "per_class.csv", index=False)
A.write_tex(T.per_class(pc), paths.tables / "per_class.tex")
P.per_class_f1(pc, paths.figures)
display(pc.sort_values("diff").round(4))

## 4. E2: Deployability (parameters, size, FLOPs, latency, memory)
Both models are measured one after the other in this same session on the same hardware. Weights do not change these numbers,
so the seed-42 checkpoints are used. Saved to `results/benchmark.json` (delete it to re-measure).

In [ ]:
from src.experiments import storage
from src.experiments.benchmark import benchmark_model
from src.experiments.qualitative import load_best

bench_file = paths.results / "benchmark.json"
class_names = sorted(manifest["label"].unique())
if bench_file.exists():
    bench = storage.load_json(bench_file)
else:
    BCFG = BenchmarkConfig() if not SMOKE else BenchmarkConfig(runs=10, warmup=2, throughput_runs=3)
    bench = {}
    for m in MODEL_ORDER:
        model = load_best(paths, run_name(m, 42), m, class_names)
        bench[m] = benchmark_model(model, m, BCFG, log=log)
        del model
    storage.save_json(bench, bench_file)
dep = A.deployability(bench)
dep.to_csv(paths.tables / "deployability.csv", index=False)
A.write_tex(T.deployability(dep), paths.tables / "deployability.tex")
display(dep.round(3).T)

## 5. E3: Accuracy versus deployment cost

In [ ]:
tr = A.tradeoff(main, dep)
tr.to_csv(paths.tables / "tradeoff.csv", index=False)
A.write_tex(T.tradeoff(tr), paths.tables / "tradeoff.tex")
display(tr.round(3))
P.tradeoff(main, dep, paths.figures);

## 6. E4: Qualitative results (confusions, examples, failure cases, Grad-CAM)

In [ ]:
tc = A.top_confusions_all(preds)
tc.to_csv(paths.tables / "top_confusions.csv", index=False)
A.write_tex(T.top_confusions(tc), paths.tables / "top_confusions.tex")
display(tc)
for m in MODEL_ORDER:
    P.confusion_errors(A.mean_confusion(preds, m), class_names, m, paths.figures)

In [ ]:
from src.experiments import qualitative as Q
pair = Q.pair_for_seed(preds, 42)
P.example_grid(Q.select_examples(pair, "correct", 12), paths.data_root, paths.figures, "e4_correct",
               "Correctly classified test images (both models; prediction and confidence)")
for m in MODEL_ORDER:
    items = Q.select_examples(pair, m, 12)
    if items:
        P.example_grid(items, paths.data_root, paths.figures, f"e4_failures_{m}",
                       f"{MODELS[m]['display']}: misclassified test images, most confident first")
items = Q.select_examples(pair, "disagree", 12)
if items:
    P.example_grid(items, paths.data_root, paths.figures, "e4_disagreements",
                   "Test images that exactly one of the two models classifies correctly")

In [ ]:
cam_models = {m: load_best(paths, run_name(m, 42), m, class_names, DEVICE) for m in MODEL_ORDER}
rows = Q.gradcam_rows(cam_models, pair, Q.gradcam_indices(pair, 3, 3), paths.data_root, DEVICE)
P.gradcam_grid(rows, paths.figures);

## 7. E5: Robustness to image corruptions (generalisation test)
All trained models (every seed) on the test set with noise, blur, low brightness, low contrast, JPEG compression and
low resolution at three severities. Each image is corrupted once, identically for every model. No retraining.

In [ ]:
from src.experiments.robustness import evaluate_corruptions, summarise_robustness
rob_file = paths.tables / "robustness_raw.csv"
if RUN_ROBUSTNESS:
    if rob_file.exists():
        rob = pd.read_csv(rob_file)
    else:
        models = {r["run"]: (r["model"], load_best(paths, r["run"], r["model"], class_names, DEVICE))
                  for _, r in runs.iterrows()}
        rob = evaluate_corruptions(models, paths, DEVICE, num_workers=CFG.num_workers, log=log)
        storage.save_dataframe(rob, rob_file)
        del models
    rsum = summarise_robustness(rob)
    rsum.to_csv(paths.tables / "robustness.csv", index=False)
    clean = {m: A.mean_std(rob[(rob.corruption == "clean") & (rob.model == m)]["accuracy"]) for m in MODEL_ORDER}
    A.write_tex(T.robustness(rsum, clean), paths.tables / "robustness.tex")
    display(rsum.round(4))
    P.robustness(rob, paths.figures);

## 8. E6: Preprocessing ablation (proposal's ImageNet mean/std vs each model's own recipe)

In [ ]:
abl_runs, _ = A.load_runs(paths, variant="imagenet")
if len(abl_runs):
    ab = A.ablation_table(runs, abl_runs)
    ab.to_csv(paths.tables / "ablation.csv", index=False)
    A.write_tex(T.ablation(ab), paths.tables / "ablation.tex")
    display(ab.round(3))
else:
    print("no ablation runs finished")

## 9. Summary (all key numbers in one file)

In [ ]:
summary = {
    "main": main.to_dict("records"), "stats": stats.to_dict("records"), "training_cost": cost.to_dict("records"),
    "deployability": dep.to_dict("records"), "tradeoff": tr.to_dict("records"),
    "robustness": rsum.to_dict("records") if RUN_ROBUSTNESS else None,
    "ablation": ab.to_dict("records") if len(abl_runs) else None,
    "environment": storage.load_json(paths.run_dir(run_name(MODEL_ORDER[0], 42)) / "config.json")["environment"],
}
storage.save_json(summary, paths.results / "summary.json")
best = main.loc[main["macro_f1_mean"].idxmax()]
print(f"Best macro-F1: {MODELS[best['model']]['display']} {best['macro_f1_mean']:.2f} ± {best['macro_f1_std']:.2f} %")
print("tables :", sorted(p.name for p in paths.tables.glob('*.tex')))
print("figures:", sorted(p.name for p in paths.figures.glob('*.pdf')))

## Appendix: source code of the experiment modules

In [ ]:
from IPython.display import Code
for f in ["config", "data", "models", "train", "evaluate", "benchmark", "robustness", "gradcam",
          "qualitative", "analysis", "latex_tables", "plots", "storage"]:
    display(Code(filename=f"src/experiments/{f}.py", language="python"))
for f in ["datasets", "preprocessing"]:
    display(Code(filename=f"src/dataset/{f}.py", language="python"))